In [2]:
import os
os.chdir('..')

In [3]:
!pwd

/Users/andreachiappo@pvh.com/Desktop/practice/deploying-machine-learning-models/assignment-section-05


In [4]:
from classification_model.config.core import config

In [5]:
from classification_model.processing.data_manager import _load_raw_dataset

In [35]:
data_ = _load_raw_dataset(file_name=config.app_configs.raw_data_file)

In [36]:
import pandas as pd

In [37]:
data = pd.DataFrame(data_)

In [38]:
from classification_model.processing.validation import validate_inputs

In [41]:
validated_data, errors = validate_inputs(input_data=data)

In [22]:
from classification_model.predict import make_prediction

In [23]:
result = make_prediction(input_data=data_)

In [25]:
from classification_model.processing.data_manager import load_pipeline

In [27]:
from classification_model import __version__ as _version

In [28]:
pipeline_file_name = f"{config.app_configs.pipeline_save_file}{_version}.pkl"
_titanic_pipe = load_pipeline(file_name=pipeline_file_name)

In [55]:
from sklearn.pipeline import Pipeline

In [56]:
from feature_engine.encoding import OneHotEncoder, RareLabelEncoder

# for imputation
from feature_engine.imputation import (
    AddMissingIndicator,
    CategoricalImputer,
    MeanMedianImputer
)
from sklearn.preprocessing import StandardScaler

from classification_model.processing.features import ExtractLetterTransformer

In [57]:
pipe = Pipeline(
    [
        # impute categorical variables with string missing
        (
            "categorical_imputation",
            CategoricalImputer(
                imputation_method="missing",
                variables=config.model_configs.categorical_vars,
            ),
        ),
        # add missing indicator to numerical variables
        (
            "missing_indicator",
            AddMissingIndicator(variables=config.model_configs.numerical_vars),
        ),
        # impute numerical variables with the median
        (
            "median_imputation",
            MeanMedianImputer(
                imputation_method="median", variables=config.model_configs.numerical_vars
            ),
        ),
        # Extract letter from cabin
        (
            "extract_letter",
            ExtractLetterTransformer(variables=config.model_configs.cabin_vars),
        ),
        # == CATEGORICAL ENCODING ======
        # remove categories present in less than 5% of the observations (0.05)
        # group them in one category called 'Rare'
        (
            "rare_label_encoder",
            RareLabelEncoder(
                tol=0.05, n_categories=1, variables=config.model_configs.categorical_vars
            ),
        ),
        # encode categorical variables using one hot encoding into k-1 variables
        (
            "categorical_encoder",
            OneHotEncoder(
                drop_last=True, variables=config.model_configs.categorical_vars
            ),
        ),
        # scale
        ("scaler", StandardScaler())
    ]
)